# Tutorial 03 — 自訂 Tool Agent
## NCHC LLM 工作坊 2026 · NVIDIA NeMo Agent Toolkit

---

### 你會學到什麼

- 怎麼用 `@register_function` 寫一個**自訂 tool**
- `nat` 如何透過 Python **entry points**（`nat.components`）發現 tool
- **description 字串**如何左右 LLM 選擇哪個 tool

### 要建的 tool

| Tool | 用途 |
|---|---|
| `temperature_converter` | 攝氏 ↔ 華氏 ↔ 克氏 |

> **開始前：** 從工作坊根目錄執行 `uv sync` 與 `uv run jupyter lab`。

---
## 步驟 1 — 確認環境


In [ ]:
import os, subprocess, getpass

result = subprocess.run(["nat", "--version"], capture_output=True, text=True)
print(result.stdout.strip() or result.stderr.strip())

if not os.environ.get("NVIDIA_API_KEY", "").startswith("nvapi-"):
    key = getpass.getpass("NVIDIA_API_KEY 尚未設定。請輸入你的 key（nvapi-...）：")
    os.environ["NVIDIA_API_KEY"] = key.strip()
print("NVIDIA_API_KEY ✓")

os.environ["NAT_CONFIG_DIR"] = os.path.abspath("nat_config")
print("NAT_CONFIG_DIR ✓")

---
## 步驟 2 — 自訂 Tool 的結構

每個 tool 都遵循這 4 個部分：

```python
# ① Config 類別 — 設定 YAML 內會用到的 _type 名稱
class MyToolConfig(FunctionBaseConfig, name="my_tool"):
    pass

# ② Decorator — 把 builder 註冊到 nat
@register_function(config_type=MyToolConfig)
async def my_tool_builder(config: MyToolConfig, builder: Builder):

    # ③ 內部 async 函式 — agent 真正會呼叫的邏輯
    async def _my_tool(text: str) -> str:
        return do_something(text)

    # ④ FunctionInfo — 對外曝露函式，並附上給 LLM 閱讀的 description
    yield FunctionInfo.from_fn(
        _my_tool,
        description="這個 tool 做什麼。範例：'100 celsius to fahrenheit'"
    )
```

> **description 是關鍵。** LLM 會讀這段字決定要呼叫**哪個** tool、以及該如何**組裝 input**。

---
## 步驟 3 — 套件結構

自訂 tool 放在一個小型 Python package 內：

```
tutorial_03_custom_tool/
├── pyproject.toml          ← 套件 metadata 與 entry point 宣告
├── configs/
│   └── config.yml          ← 引用自訂 tool 的 workflow config
└── src/
    └── temperature_converter/
        ├── __init__.py
        └── register.py     ← temperature_converter 的定義
```

先看關鍵檔案的內容，再安裝套件。

---
## 步驟 4 — 安裝與驗證

看完程式碼後，把 package 裝進這個 notebook 的 kernel。
`nat` 會透過 `pyproject.toml` 宣告的 `nat.components` entry point 自動發現這個 tool。

In [ ]:
%pip install tutorial_03_custom_tool/

---
## 步驟 5 — 驗證註冊

確認 `nat` 看得到剛安裝的 tool：

```python
import importlib.metadata
eps = importlib.metadata.entry_points().select(group="nat.components")
for ep in eps:
    if "temperature" in ep.name:
        print(f"Found: {ep.name}  →  {ep.value}")
```

然後執行 agent：

In [ ]:
!nat run \
    --config_file tutorial_03_custom_tool/configs/config.yml \
    --input "把 100 攝氏度換算成華氏度。"

In [ ]:
!nat run \
    --config_file tutorial_03_custom_tool/configs/config.yml \
    --input "37 攝氏度等於多少華氏度？那是人體的正常體溫。"

In [ ]:
!nat run \
    --config_file tutorial_03_custom_tool/configs/config.yml \
    --input "水在攝氏 100 度沸騰、攝氏 0 度結冰，換算成華氏分別是幾度？"

---
## 總結

| 概念 | 學到了什麼 |
|---|---|
| **`@register_function`** | 把 Python 函式接到 nat 的 decorator |
| **`FunctionInfo.from_fn`** | 把函式與 LLM 用來挑 tool 的 description 綁在一起 |
| **`FunctionBaseConfig(name=...)`** | 設定 YAML config 內使用的 `_type` 名稱 |
| **`nat.components` entry point** | `nat` 啟動時用來發現已安裝外掛的機制 |

### 關鍵觀念

`FunctionInfo.from_fn` 內的 **description 字串**是 LLM 用來挑 tool 的依據。
一段精確、附範例的描述跟 tool 本身的實作一樣重要。

---

🎉 **工作坊完成！** 你建了三個 NeMo Agent Toolkit workflow：
- 簡單的時間 agent（Tutorial 01）
- 用 ReAct 推理的維基百科研究 agent（Tutorial 02）
- 自訂的溫度轉換 tool（Tutorial 03）